# NexusRAG — 03: Evaluation

Demonstrates all 3 metric layers: deterministic, LLM judge, and aggregate reporting.

In [ ]:
import sys
sys.path.insert(0, '..')
import asyncio

In [ ]:
# Layer 1: Deterministic metrics
from src.evaluation.metrics import (
    context_precision, context_recall, mrr, ndcg_at_k,
    cosine_similarity, latency_stats
)

retrieved = ['chunk_1', 'chunk_2', 'chunk_3', 'chunk_4', 'chunk_5']
relevant  = {'chunk_1', 'chunk_3', 'chunk_5'}

print('=== Layer 1: Deterministic Metrics ===')
print(f'Context Precision : {context_precision(retrieved, relevant):.4f}')  # 3/5 = 0.60
print(f'Context Recall    : {context_recall(retrieved, relevant):.4f}')     # 3/3 = 1.00
print(f'MRR               : {mrr(retrieved, relevant):.4f}')               # 1/1 = 1.00
print(f'NDCG@5            : {ndcg_at_k(retrieved, relevant, k=5):.4f}')

latencies = [120, 250, 180, 310, 450, 520, 200, 290, 170, 380]
stats = latency_stats(latencies)
print(f'\nLatency p50={stats.p50:.0f}ms  p95={stats.p95:.0f}ms  p99={stats.p99:.0f}ms')

In [ ]:
# Layer 2: LLM Judge (requires Ollama)
from src.evaluation.llm_judge import LLMJudge

judge = LLMJudge()
result = await judge.evaluate(
    query='What is machine learning?',
    context='Machine learning is a subset of artificial intelligence that enables systems to learn from data without being explicitly programmed.',
    answer='Machine learning allows computers to learn from data automatically.',
    ground_truth='Machine learning is a subset of AI that learns from data.'
)
print('=== Layer 2: LLM Judge ===')
print(f'Faithfulness : {result.faithfulness_score:.3f}')
print(f'Relevancy    : {result.relevancy_score:.3f}')
print(f'Completeness : {result.completeness_score:.3f}')
print(f'Hallucinated : {result.hallucinated_claims}')

In [ ]:
# Layer 3: Full benchmark run
from src.evaluation.benchmark import BenchmarkRunner
from src.evaluation.report_generator import ReportGenerator

runner = BenchmarkRunner(use_llm_judge=False)  # deterministic only for speed
cases = runner.load_test_cases()
print(f'Loaded {len(cases)} test cases')
print(f'Difficulties: {set(c.difficulty for c in cases)}')
print(f'Tiers: {set(c.expected_tier for c in cases)}')

In [ ]:
# Generate report + charts
import json

# Simulate a report for visualization
mock_report = {
    'run_id': 'demo',
    'total_cases': 50,
    'mean_context_precision': 0.72,
    'mean_context_recall': 0.68,
    'mean_faithfulness': 0.81,
    'mean_relevancy': 0.85,
    'mean_completeness': 0.74,
    'latency': {'p50': 320, 'p95': 1250, 'p99': 2100, 'mean': 480},
    'tier_distribution': {'TIER_1': 38, 'TIER_2': 9, 'TIER_3': 3},
    'per_case': [
        {'index': i, 'query': f'Query {i}', 'difficulty': 'medium',
         'expected_tier': 'TIER_1', 'actual_tier': 'TIER_1',
         'latency_ms': 200 + i*10,
         'det_context_precision': 0.6 + (i % 5) * 0.05,
         'det_context_recall': 0.65 + (i % 4) * 0.05,
         'llm_faithfulness_score': 0.75 + (i % 3) * 0.05,
         'answer': 'Sample answer', 'error': None}
        for i in range(50)
    ]
}

gen = ReportGenerator()
md_path, charts = gen.generate(mock_report, 'demo')
print(f'Report: {md_path}')
print(f'Charts: {charts}')

In [ ]:
# Display charts inline
from IPython.display import Image, display
for chart in charts:
    display(Image(chart))